# Chapter 3: RDF Data and Loading

**Level 1: Foundations** · ⏱ about 2 hours · Dataset: *Recipes & Nutrition*

### What you will learn

1. The three kinds of RDF terms: **IRIs, literals and blank nodes**
2. **Datatypes** and **language tags**, and why a number stored as text causes wrong answers
3. When to use **blank nodes**, and the trap they set when you reload data
4. The main **serialization formats**: Turtle, N-Triples, JSON-LD, RDF/XML, TriG, N-Quads
5. **Named graphs** vs the **default graph**, and how queries choose between them
6. **Loading data** from Python: `File`, `Raw`, content types, compression, and transactions that fail safely
7. **Hands-on:** convert the Recipes CSV files into RDF, load them into Stardog and export them as JSON-LD

This chapter introduces the **Recipes & Nutrition** dataset used in Chapters 3 to 9. The hands-on writes it to `data/recipes/recipes.ttl`, which later chapters load directly.

In [1]:
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog
from rdflib import BNode, Dataset, Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, SKOS, XSD

from kg import client

conn = client.connect()                 # one connection for the notebook, closed at the end
PREFIX = 'urn:tutorial:ch03'            # all of this chapter's named graphs start with this

def select_df(query, **kwargs):
    result = conn.select(query, **kwargs)
    cols = result['head']['vars']
    return pd.DataFrame([{c: b[c]['value'] if c in b else None for c in cols}
                         for b in result['results']['bindings']], columns=cols)

def load_turtle(text, graph):
    '''Replace the contents of `graph` with the given Turtle.'''
    with client.transaction(conn):
        conn.clear(graph_uri=graph)
        conn.add(stardog.content.Raw(text.encode(), 'text/turtle'), graph_uri=graph)

print('Connected to', client.load_settings().database)

Connected to mysampledb


`client.transaction(conn)` is a helper from `src/kg/client.py`. It does `begin()` → your code → `commit()`, and rolls back if something fails. Section 3.6 shows why it's needed.

## 3.1 The three kinds of RDF terms

| Term | Example | Can be subject? | Can be object? | Identity |
|---|---|---|---|---|
| **IRI** | `<http://example.org/recipes/recipe/pad-thai>` or `recipe:pad-thai` | ✅ | ✅ | Global: the same IRI means the same thing everywhere |
| **Literal** | `"Pad Thai"`, `25`, `"2025-02-21"^^xsd:date`, `"Pad thaï"@fr` | ❌ | ✅ | Its value |
| **Blank node** | `[ rec:quantity 200 ]` or `_:b1` | ✅ | ✅ | Local to one file or load, with no global name |

Predicates are always IRIs.

## 3.2 Literals: datatypes and language tags

A literal is a value with a **datatype**. Without one it's a plain string (`xsd:string`).

In [2]:
examples = [
    Literal('Pad Thai'),
    Literal(25),
    Literal('25'),
    Literal(4.5),
    Literal('4.50', datatype=XSD.decimal),
    Literal(True),
    Literal('2025-02-21', datatype=XSD.date),
    Literal('2025-02-21T18:30:00', datatype=XSD.dateTime),
    Literal('Pad thaï', lang='fr'),
]
nm = Graph().namespace_manager           # knows the standard prefixes (xsd:, rdf:, …)
pd.DataFrame([(l.n3(nm),
               l.datatype.n3(nm) if l.datatype else ('rdf:langString' if l.language else 'xsd:string'),
               type(l.toPython()).__name__)
              for l in examples], columns=['Turtle', 'datatype', 'Python value'])

,Turtle,datatype,Python value
0,"""Pad Thai""",xsd:string,str
1,"""25""^^xsd:integer",xsd:integer,int
2,"""25""",xsd:string,str
3,"""4.5""^^xsd:double",xsd:double,float
4,"""4.50""^^xsd:decimal",xsd:decimal,Decimal
5,"""true""^^xsd:boolean",xsd:boolean,bool
6,"""2025-02-21""^^xsd:date",xsd:date,date
7,"""2025-02-21T18:30:00""^^xsd:dateTime",xsd:dateTime,datetime
8,"""Pad thaï""@fr",rdf:langString,str


| Datatype | Use for | Turtle |
|---|---|---|
| `xsd:string` | Text | `"Pad Thai"` |
| `xsd:integer` | Whole numbers | `25` |
| `xsd:decimal` | Exact decimals (money, grams) | `4.50` |
| `xsd:double` | Floating point (measurements, scores) | `4.5e0` |
| `xsd:boolean` | true / false | `true` |
| `xsd:date`, `xsd:dateTime` | Dates and timestamps | `"2025-02-21"^^xsd:date` |
| `rdf:langString` | Text in a specific language | `"Pad thaï"@fr` |

### Why datatypes matter

CSV files have no types, so it's easy to load numbers as text. Here are prep times stored both ways:

In [3]:
TYPES_GRAPH = f'{PREFIX}:types'
load_turtle('''
@prefix ex: <http://example.org/demo#> .
ex:soup   ex:minutesText "40" ; ex:minutes 40 .
ex:salmon ex:minutesText "20" ; ex:minutes 20 .
ex:pesto  ex:minutesText "9"  ; ex:minutes 9 .
''', TYPES_GRAPH)

def sorted_by(prop):
    return select_df(f'''PREFIX ex: <http://example.org/demo#>
        SELECT ?dish ?value FROM <{TYPES_GRAPH}> {{ ?dish ex:{prop} ?value }} ORDER BY ?value''')

display(sorted_by('minutesText'), sorted_by('minutes'))

,dish,value
0,http://example.org/demo#salmon,20
1,http://example.org/demo#soup,40
2,http://example.org/demo#pesto,9


,dish,value
0,http://example.org/demo#pesto,9
1,http://example.org/demo#salmon,20
2,http://example.org/demo#soup,40


Stored as **text**, "9" sorts *after* "40", because strings compare character by character, like words in a dictionary. As **integers** the order is correct. Comparisons go wrong the same way:

In [4]:
for prop in ('minutesText', 'minutes'):
    quick = select_df(f'''PREFIX ex: <http://example.org/demo#>
        SELECT ?dish FROM <{TYPES_GRAPH}> {{ ?dish ex:{prop} ?m  FILTER(?m < 30) }}''')
    print(f'{prop:12} under 30 minutes: {sorted(quick.dish.str.split("#").str[1])}')

minutesText  under 30 minutes: []


minutes      under 30 minutes: ['pesto', 'salmon']


With text values, `?m < 30` compares a string with a number. That comparison is an error in SPARQL, so the filter silently drops **every** row. Store numbers as numbers, dates as `xsd:date`, and so on.

> 🔁 **Neo4j equivalent:** `LOAD CSV` also reads every value as a string, and you convert with `toInteger()` / `date()`. Forgetting leads to the same bugs.

### Language tags

A literal can carry a **language tag**, so one property can hold the same text in several languages. Queries pick one with `lang()` or `langMatches()`:

In [5]:
LANG_GRAPH = f'{PREFIX}:lang'
load_turtle('''
@prefix ex: <http://example.org/demo#> .
ex:soup  ex:name "Vegetable Soup"@en , "Soupe de légumes"@fr , "Gemüsesuppe"@de .
ex:pesto ex:name "Pesto Pasta"@en , "Pâtes au pesto"@fr .
''', LANG_GRAPH)

select_df(f'''PREFIX ex: <http://example.org/demo#>
SELECT ?dish ?french ?anyGerman
FROM <{LANG_GRAPH}>
WHERE {{
  ?dish ex:name ?french  FILTER(lang(?french) = "fr")
  OPTIONAL {{ ?dish ex:name ?anyGerman  FILTER(langMatches(lang(?anyGerman), "de")) }}
}}''')

,dish,french,anyGerman
0,http://example.org/demo#soup,Soupe de légumes,Gemüsesuppe
1,http://example.org/demo#pesto,Pâtes au pesto,NaN


`"Soupe"@fr` and `"Soupe"` are **different** literals. A query for `ex:name "Vegetable Soup"` does not match `"Vegetable Soup"@en`. Stick to one convention per property.

## 3.3 Blank nodes

A **blank node** is a node without a global name. In Turtle, `[ … ]` creates one. Blank nodes are handy for small structured values that don't deserve their own identity, like an ingredient line "200 g of spaghetti":

```turtle
recipe:pesto-pasta rec:hasIngredientLine [
    rec:ingredient ingredient:spaghetti ;
    rec:quantity 200 ;
    rec:unit "g"
] .
```

Without a blank node we'd have to invent an IRI for every line.

In [6]:
g = Graph()
g.parse(data='''
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix ingredient: <http://example.org/recipes/ingredient/> .
recipe:pesto-pasta rec:hasIngredientLine
    [ rec:ingredient ingredient:spaghetti ; rec:quantity 200 ; rec:unit "g" ] ,
    [ rec:ingredient ingredient:basil     ; rec:quantity 40  ; rec:unit "g" ] .
''', format='turtle')
for s, p, o in sorted(g):
    print(f'{s.n3(g.namespace_manager):22} {p.n3(g.namespace_manager):26} {o.n3(g.namespace_manager)}')

_:n885277d690b2432cab97a1e856121f55b1 rec:ingredient             ingredient:spaghetti
_:n885277d690b2432cab97a1e856121f55b1 rec:quantity               "200"^^xsd:integer
_:n885277d690b2432cab97a1e856121f55b1 rec:unit                   "g"
_:n885277d690b2432cab97a1e856121f55b2 rec:ingredient             ingredient:basil
_:n885277d690b2432cab97a1e856121f55b2 rec:quantity               "40"^^xsd:integer
_:n885277d690b2432cab97a1e856121f55b2 rec:unit                   "g"
recipe:pesto-pasta     rec:hasIngredientLine      _:n885277d690b2432cab97a1e856121f55b1
recipe:pesto-pasta     rec:hasIngredientLine      _:n885277d690b2432cab97a1e856121f55b2


The parser gave each `[ … ]` a generated local ID (`_:N…`). Those IDs are **new every time the data is parsed or loaded**. That leads to a real trap:

### The reload trap

RDF graphs are **sets**, so loading a triple that already exists changes nothing. But a blank node from a second load is a *new* node, so its triples are *new* triples:

In [7]:
BNODE_GRAPH = f'{PREFIX}:bnodes'
TTL = '''
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix ingredient: <http://example.org/recipes/ingredient/> .
recipe:pesto-pasta rec:name "Pesto Pasta" ;
    rec:hasIngredientLine [ rec:ingredient ingredient:spaghetti ; rec:quantity 200 ; rec:unit "g" ] .
'''
def counts():
    return select_df(f'''PREFIX rec: <http://example.org/recipes#>
        SELECT ?triples ?ingredient_lines FROM <{BNODE_GRAPH}> WHERE {{
          {{ SELECT (COUNT(*) AS ?triples) {{ ?s ?p ?o }} }}
          {{ SELECT (COUNT(?line) AS ?ingredient_lines) {{ ?r rec:hasIngredientLine ?line }} }}
        }}''')

load_turtle(TTL, BNODE_GRAPH)                    # clears the graph, then loads
first = counts()
with client.transaction(conn):                   # load the same file again WITHOUT clearing
    conn.add(stardog.content.Raw(TTL.encode(), 'text/turtle'), graph_uri=BNODE_GRAPH)
pd.concat([first, counts()], keys=['after 1st load', 'after 2nd load']).droplevel(1)

,triples,ingredient_lines
after 1st load,5,1
after 2nd load,9,2


The name triple was not duplicated, but the recipe now has **two** identical-looking ingredient lines.

**Two fixes:**

1. **Make loads idempotent**: clear the target graph before reloading, as `load_turtle` does. This is the pattern used throughout the tutorial.
2. **Give the nodes IRIs** (called *skolemization*), e.g. `recipe:pesto-pasta/line/1`. Then reloading can't duplicate them, and other data can point at a line.

**Use blank nodes** for small value-like structures that are always loaded together with their parent. **Use IRIs** for anything that's updated separately or referenced from elsewhere.

> 🔁 **Neo4j equivalent:** a blank node is like an anonymous node you `CREATE` without a unique key. Running the script twice creates duplicates unless you use `MERGE` on a key, which is the same lesson.

## 3.4 Serialization formats

One RDF graph can be written in many formats. They all mean exactly the same triples:

In [8]:
small = Graph()
small.bind('rec', 'http://example.org/recipes#')
small.bind('recipe', 'http://example.org/recipes/recipe/')
small.parse(data='''
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
recipe:pesto-pasta a rec:Recipe ; rec:name "Pesto Pasta"@en ; rec:prepMinutes 15 .
''', format='turtle')

for fmt in ('turtle', 'nt', 'json-ld', 'xml'):
    print(f'=== {fmt} ===')
    print(small.serialize(format=fmt).strip(), '\n')

=== turtle ===
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

recipe:pesto-pasta a rec:Recipe ;
    rec:name "Pesto Pasta"@en ;
    rec:prepMinutes 15 . 

=== nt ===
<http://example.org/recipes/recipe/pesto-pasta> <http://example.org/recipes#name> "Pesto Pasta"@en .
<http://example.org/recipes/recipe/pesto-pasta> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://example.org/recipes#Recipe> .
<http://example.org/recipes/recipe/pesto-pasta> <http://example.org/recipes#prepMinutes> "15"^^<http://www.w3.org/2001/XMLSchema#integer> . 

=== json-ld ===
[
  {
    "@id": "http://example.org/recipes/recipe/pesto-pasta",
    "@type": [
      "http://example.org/recipes#Recipe"
    ],
    "http://example.org/recipes#name": [
      {
        "@language": "en",
        "@value": "Pesto Pasta"
      }
    ],
    "http://example.org/recipes#prepMinutes": [
      {
        "@type": "http://

**Named graphs need a quad format.** Turtle, N-Triples, JSON-LD (in this simple form) and RDF/XML hold one graph. **TriG** and **N-Quads** add a fourth element, the graph each triple belongs to:

In [9]:
ds = Dataset()
ds.bind('rec', 'http://example.org/recipes#')
ds.bind('recipe', 'http://example.org/recipes/recipe/')
kitchen = ds.graph(URIRef('urn:example:kitchen'))
nutrition = ds.graph(URIRef('urn:example:nutrition'))
REC = Namespace('http://example.org/recipes#')
PESTO = URIRef('http://example.org/recipes/recipe/pesto-pasta')
kitchen.add((PESTO, REC.prepMinutes, Literal(15)))
nutrition.add((PESTO, REC.kcalPerServing, Literal(620)))

print('=== trig ===');    print(ds.serialize(format='trig').strip(), '\n')
print('=== nquads ==='); print(ds.serialize(format='nquads').strip())

=== trig ===
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<urn:example:nutrition> {
    recipe:pesto-pasta rec:kcalPerServing 620 .
}

<urn:example:kitchen> {
    recipe:pesto-pasta rec:prepMinutes 15 .
} 

=== nquads ===
<http://example.org/recipes/recipe/pesto-pasta> <http://example.org/recipes#kcalPerServing> "620"^^<http://www.w3.org/2001/XMLSchema#integer> <urn:example:nutrition> .
<http://example.org/recipes/recipe/pesto-pasta> <http://example.org/recipes#prepMinutes> "15"^^<http://www.w3.org/2001/XMLSchema#integer> <urn:example:kitchen> .


| Format | Extension | Graphs | Best for |
|---|---|---|---|
| **Turtle** | `.ttl` | 1 | Writing by hand, reading, git diffs. The default choice |
| **N-Triples** | `.nt` | 1 | Huge files: one triple per line, easy to split and stream |
| **JSON-LD** | `.jsonld` | 1+ | Web APIs and JavaScript; looks like normal JSON |
| **RDF/XML** | `.rdf`, `.owl` | 1 | Legacy tools and older ontologies (e.g. from Protégé) |
| **TriG** | `.trig` | many | Turtle plus named graphs |
| **N-Quads** | `.nq` | many | N-Triples plus named graphs; bulk exports |

Stardog reads and writes all of these.

## 3.5 Named graphs and the default graph

A Stardog database is a **dataset**: one **default graph** (triples with no graph name) plus any number of **named graphs**.

Named graphs are how you keep data **apart and traceable**:

- by **source**: CRM data, ERP data, scraped data
- by **version or date**: a snapshot per month
- by **purpose**: model vs data, as Chapter 7 will do
- by **project or chapter**: exactly how this tutorial isolates each chapter

Let's load a TriG file with two sources describing the same recipe. With TriG, the graph names come **from the file**, so no `graph_uri` is passed:

In [10]:
KITCHEN, NUTRITION = f'{PREFIX}:source:kitchen', f'{PREFIX}:source:nutrition'
TRIG = f'''
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
<{KITCHEN}>   {{ recipe:pesto-pasta rec:prepMinutes 15 ; rec:servings 2 . }}
<{NUTRITION}> {{ recipe:pesto-pasta rec:kcalPerServing 620 . }}
'''
with client.transaction(conn):
    conn.clear(graph_uri=KITCHEN)
    conn.clear(graph_uri=NUTRITION)
    conn.add(stardog.content.Raw(TRIG.encode(), 'application/trig'))

Q = '''PREFIX rec: <http://example.org/recipes#>
SELECT ?property ?value {dataset} WHERE {{ <http://example.org/recipes/recipe/pesto-pasta> ?property ?value }}'''
for label, dataset in [('no FROM (default graph only)', ''),
                       ('FROM kitchen', f'FROM <{KITCHEN}>'),
                       ('FROM kitchen + FROM nutrition', f'FROM <{KITCHEN}> FROM <{NUTRITION}>')]:
    print(f'--- {label}')
    df = select_df(Q.format(dataset=dataset))
    print(df.to_string(index=False) if len(df) else '(no rows)')

--- no FROM (default graph only)


(no rows)
--- FROM kitchen


                              property value
http://example.org/recipes#prepMinutes    15
   http://example.org/recipes#servings     2
--- FROM kitchen + FROM nutrition


                                 property value
   http://example.org/recipes#prepMinutes    15
      http://example.org/recipes#servings     2
http://example.org/recipes#kcalPerServing   620


- **No `FROM`:** Stardog reads only the **default graph**, where there's nothing, because the database option `query.all.graphs` is `false` (Chapter 2).
- **`FROM <g>`:** read that graph.
- **Several `FROM`s:** Stardog **merges** them into one graph for the query. The recipe's facts from both sources come together.

To see **which graph each triple came from**, use `FROM NAMED` with a `GRAPH ?g` pattern:

In [11]:
select_df(f'''PREFIX rec: <http://example.org/recipes#>
SELECT ?graph ?property ?value
FROM NAMED <{KITCHEN}> FROM NAMED <{NUTRITION}>
WHERE {{ GRAPH ?graph {{ <http://example.org/recipes/recipe/pesto-pasta> ?property ?value }} }}''')

,graph,property,value
0,urn:tutorial:ch03:source:kitchen,http://example.org/recipes#prepMinutes,15
1,urn:tutorial:ch03:source:kitchen,http://example.org/recipes#servings,2
2,urn:tutorial:ch03:source:nutrition,http://example.org/recipes#kcalPerServing,620


Stardog also has **special graph names**:

| Graph IRI | Means |
|---|---|
| `tag:stardog:api:context:default` | The default graph |
| `tag:stardog:api:context:named` | All named graphs, merged |
| `tag:stardog:api:context:all` | Default graph + all named graphs |

`FROM <tag:stardog:api:context:all>` is handy for exploring, but in applications name the graphs you mean, so other data in the database can't leak into your results.

> 🔁 **Neo4j equivalent:** property graphs have no named graphs. The nearest options are separate databases or an extra label or property on every node, and neither records provenance per relationship.

## 3.6 Loading data from Python

`conn.add(content, graph_uri=...)` takes a **content object**:

| Class | Use | Example |
|---|---|---|
| `stardog.content.File` | A local file; format and compression detected from the extension | `File('data.ttl')`, `File('dump.nt.gz')` |
| `stardog.content.Raw` | Bytes or text in memory; give the content type | `Raw(text, 'text/turtle')` |
| `stardog.content.URL` | The **Stardog server** downloads a URL | `URL('https://…/data.ttl')` |

Common content types: `text/turtle`, `application/n-triples`, `application/ld+json`, `application/rdf+xml`, `application/trig`, `application/n-quads`.

### Loads happen in transactions

`add` only works between `begin()` and `commit()`. If anything fails in between, **nothing** is saved. Let's load a broken file:

In [12]:
BROKEN = 'ex:bad ex:missing .'           # prefix 'ex' is never declared, and the object is missing
try:
    with client.transaction(conn):
        conn.add(stardog.content.Raw(BROKEN.encode(), 'text/turtle'), graph_uri=f'{PREFIX}:broken')
except stardog.exceptions.StardogException as e:
    print(f'Load failed: HTTP {e.http_code}: {e}')

print('Nothing was saved:', not conn.ask(f'ASK {{ GRAPH <{PREFIX}:broken> {{ ?s ?p ?o }} }}'))
print('Connection still usable, e.g. size():', conn.size())

Load failed: HTTP 400: [400] QEIVR2: Namespace prefix 'ex' used but not defined [L1]


Nothing was saved: True


Connection still usable, e.g. size(): 2540


The error message even gives the line (`[L1]`).

**Why the helper?** When a request inside a transaction fails, Stardog **aborts the transaction on the server** straight away. pystardog doesn't notice: its `rollback()` then fails with *"not a known transaction"*, and the connection refuses to `begin()` a new one. `client.transaction()` handles that, so after a failed load the connection keeps working.

Plain pystardog code looks like this, and is fine as long as nothing fails:

```python
conn.begin()
conn.add(stardog.content.File('data.ttl'), graph_uri='urn:my:graph')
conn.commit()
```

## 3.7 Hands-on: the Recipes dataset from CSV to RDF

The raw data is six CSV files in `data/recipes/csv/`:

In [13]:
CSV = Path('..', 'data', 'recipes', 'csv')
tables = {name: pd.read_csv(CSV / f'{name}.csv')
          for name in ('recipes', 'ingredients', 'categories', 'allergens',
                       'recipe_ingredients', 'ingredient_allergens')}
for name, df in tables.items():
    print(f'{name:22} {len(df):3} rows   columns: {", ".join(df.columns)}')
tables['recipes'].head(3)

recipes                 14 rows   columns: recipe_id, name, name_fr, cuisine, prep_minutes, servings, created, description
ingredients             42 rows   columns: ingredient_id, name, category_id, kcal_per_100g, protein_g_per_100g
categories              28 rows   columns: category_id, name, parent_id
allergens               14 rows   columns: allergen_name
recipe_ingredients      84 rows   columns: recipe_id, ingredient_id, quantity, unit
ingredient_allergens    27 rows   columns: ingredient_id, allergen_name


,recipe_id,name,name_fr,cuisine,prep_minutes,servings,created,description
0,R01,Spaghetti Carbonara,Spaghetti à la carbonara,Italian,20,2,2025-01-14,"Italian pasta tossed with egg, parmesan and cr..."
1,R02,Margherita Pizza,Pizza Margherita,Italian,30,2,2025-02-03,"Thin-crust pizza with tomato, mozzarella and f..."
2,R03,Pad Thai,Pad thaï,Thai,25,2,2025-02-21,"Stir-fried rice noodles with shrimp, tofu, egg..."


### Step 1: design the IRIs

Good IRIs are **stable** (they don't change when a name is edited later), **readable** and **grouped by kind**. We use one namespace for the vocabulary (classes and properties) and one per kind of thing:

| Prefix | Namespace | Example |
|---|---|---|
| `rec:` | `http://example.org/recipes#` | `rec:Recipe`, `rec:hasIngredient` (vocabulary) |
| `recipe:` | `http://example.org/recipes/recipe/` | `recipe:pad-thai` |
| `ingredient:` | `http://example.org/recipes/ingredient/` | `ingredient:parmesan-cheese` |
| `category:` | `http://example.org/recipes/category/` | `category:cheese` |
| `allergen:` | `http://example.org/recipes/allergen/` | `allergen:tree-nuts` |

Local names are **slugs** (lower-case, hyphens) made from the names. The original IDs stay available as properties (`rec:recipeId "R03"`).

### Step 2: decide the shape

```
recipe:pad-thai  a rec:Recipe ;
    rec:name "Pad Thai"@en , "Pad thaï"@fr ;          # language tags
    rec:cuisine "Thai" ; rec:prepMinutes 25 ;          # typed literals
    rec:servings 2 ; rec:created "2025-02-21"^^xsd:date ;
    rec:hasIngredient ingredient:shrimp , … ;          # direct link, simple to query
    rec:hasIngredientLine [ rec:ingredient ingredient:shrimp ;   # details, as blank nodes
                            rec:quantity 150 ; rec:unit "g" ] , … .

ingredient:shrimp a rec:Ingredient ; rec:name "Shrimp" ;
    rec:inCategory category:shellfish ; rec:hasAllergen allergen:crustaceans ;
    rec:kcalPer100g 99.0 ; rec:proteinPer100g 24.0 .

category:shellfish a rec:Category ; rec:name "Shellfish" ;
    skos:broader category:seafood .                    # hierarchy, from the standard SKOS vocabulary
```

We keep **both** `rec:hasIngredient` (easy queries) and `rec:hasIngredientLine` (quantities). It's a common modelling trade-off: some redundancy for simpler queries.

### Step 3: convert

In [14]:
import re

REC        = Namespace('http://example.org/recipes#')
RECIPE     = Namespace('http://example.org/recipes/recipe/')
INGREDIENT = Namespace('http://example.org/recipes/ingredient/')
CATEGORY   = Namespace('http://example.org/recipes/category/')
ALLERGEN   = Namespace('http://example.org/recipes/allergen/')

def slug(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')

kg = Graph()
for prefix, ns in [('rec', REC), ('recipe', RECIPE), ('ingredient', INGREDIENT),
                   ('category', CATEGORY), ('allergen', ALLERGEN), ('skos', SKOS)]:
    kg.bind(prefix, ns)

# Categories and their hierarchy
for row in tables['categories'].itertuples():
    cat = CATEGORY[row.category_id]
    kg.add((cat, RDF.type, REC.Category))
    kg.add((cat, REC.name, Literal(row.name)))
    if isinstance(row.parent_id, str):
        kg.add((cat, SKOS.broader, CATEGORY[row.parent_id]))

# Allergens
for row in tables['allergens'].itertuples():
    kg.add((ALLERGEN[slug(row.allergen_name)], RDF.type, REC.Allergen))
    kg.add((ALLERGEN[slug(row.allergen_name)], REC.name, Literal(row.allergen_name)))

# Ingredients, with typed nutrition values
ingredient_iri = {}
for row in tables['ingredients'].itertuples():
    ing = INGREDIENT[slug(row.name)]
    ingredient_iri[row.ingredient_id] = ing
    kg.add((ing, RDF.type, REC.Ingredient))
    kg.add((ing, REC.ingredientId, Literal(row.ingredient_id)))
    kg.add((ing, REC.name, Literal(row.name)))
    kg.add((ing, REC.inCategory, CATEGORY[row.category_id]))
    kg.add((ing, REC.kcalPer100g, Literal(row.kcal_per_100g, datatype=XSD.decimal)))
    kg.add((ing, REC.proteinPer100g, Literal(row.protein_g_per_100g, datatype=XSD.decimal)))

for row in tables['ingredient_allergens'].itertuples():
    kg.add((ingredient_iri[row.ingredient_id], REC.hasAllergen, ALLERGEN[slug(row.allergen_name)]))

# Recipes: language-tagged names, typed values
recipe_iri = {}
for row in tables['recipes'].itertuples():
    r = RECIPE[slug(row.name)]
    recipe_iri[row.recipe_id] = r
    kg.add((r, RDF.type, REC.Recipe))
    kg.add((r, REC.recipeId, Literal(row.recipe_id)))
    kg.add((r, REC.name, Literal(row.name, lang='en')))
    kg.add((r, REC.name, Literal(row.name_fr, lang='fr')))
    kg.add((r, REC.cuisine, Literal(row.cuisine)))
    kg.add((r, REC.prepMinutes, Literal(int(row.prep_minutes))))
    kg.add((r, REC.servings, Literal(int(row.servings))))
    kg.add((r, REC.created, Literal(row.created, datatype=XSD.date)))
    kg.add((r, REC.description, Literal(row.description, lang='en')))

# Ingredient lines as blank nodes, plus the direct hasIngredient link
for row in tables['recipe_ingredients'].itertuples():
    r, ing = recipe_iri[row.recipe_id], ingredient_iri[row.ingredient_id]
    line = BNode()
    kg.add((r, REC.hasIngredient, ing))
    kg.add((r, REC.hasIngredientLine, line))
    kg.add((line, RDF.type, REC.IngredientLine))
    kg.add((line, REC.ingredient, ing))
    kg.add((line, REC.quantity, Literal(row.quantity, datatype=XSD.decimal)))
    kg.add((line, REC.unit, Literal(row.unit)))

print(f'{len(kg)} triples')

1020 triples


### Step 4: check the result

Always check a conversion before loading it. Count things by type, and look at one complete recipe:

In [15]:
counts = kg.query('''SELECT ?type (COUNT(?s) AS ?n) WHERE { ?s a ?type } GROUP BY ?type ORDER BY DESC(?n)''')
display(pd.DataFrame([(t.n3(kg.namespace_manager), int(n)) for t, n in counts], columns=['type', 'count']))

one = Graph(namespace_manager=kg.namespace_manager)
pad_thai = RECIPE['pad-thai']
for t in kg.triples((pad_thai, None, None)):
    one.add(t)
    if t[1] == REC.hasIngredientLine:
        for line_triple in kg.triples((t[2], None, None)):
            one.add(line_triple)
print('\n'.join(one.serialize(format='turtle').splitlines()[:30]), '\n…')

,type,count
0,rec:IngredientLine,84
1,rec:Ingredient,42
2,rec:Category,28
3,rec:Allergen,14
4,rec:Recipe,14


@prefix ingredient: <http://example.org/recipes/ingredient/> .
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

recipe:pad-thai a rec:Recipe ;
    rec:created "2025-02-21"^^xsd:date ;
    rec:cuisine "Thai" ;
    rec:description "Stir-fried rice noodles with shrimp, tofu, egg and peanuts."@en ;
    rec:hasIngredient ingredient:egg,
        ingredient:fish-sauce,
        ingredient:garlic,
        ingredient:lime,
        ingredient:peanuts,
        ingredient:rice-noodles,
        ingredient:shrimp,
        ingredient:tofu ;
    rec:hasIngredientLine [ a rec:IngredientLine ;
            rec:ingredient ingredient:tofu ;
            rec:quantity 100.0 ;
            rec:unit "g" ],
        [ a rec:IngredientLine ;
            rec:ingredient ingredient:fish-sauce ;
            rec:quantity 30.0 ;
            rec:unit "ml" ],
        [ a rec:IngredientLine ;
            rec:ingredient ingr

The counts match the CSVs: 14 recipes, 42 ingredients, 28 categories, 14 allergens and 84 ingredient lines.

### Step 5: save it for the next chapters

In [16]:
TTL_FILE = Path('..', 'data', 'recipes', 'recipes.ttl')
TTL_FILE.write_text(
    '# Recipes & Nutrition dataset, generated from data/recipes/csv by Chapter 3.\n'
    + kg.serialize(format='turtle'), encoding='utf-8')
print(f'Wrote {TTL_FILE} ({TTL_FILE.stat().st_size:,} bytes)')

Wrote ..\data\recipes\recipes.ttl (33,504 bytes)


### Step 6: load it into Stardog

`File` detects Turtle from the `.ttl` extension. We load it into this chapter's graph with the transaction helper:

In [17]:
RECIPES_GRAPH = f'{PREFIX}:recipes'

with client.transaction(conn):
    conn.clear(graph_uri=RECIPES_GRAPH)                       # idempotent: safe to rerun
    conn.add(stardog.content.File(str(TTL_FILE)), graph_uri=RECIPES_GRAPH)

select_df(f'''
SELECT ?type (COUNT(?s) AS ?count)
FROM <{RECIPES_GRAPH}>
WHERE {{ ?s a ?type }}
GROUP BY ?type ORDER BY DESC(?count)''')

,type,count
0,http://example.org/recipes#IngredientLine,84
1,http://example.org/recipes#Ingredient,42
2,http://example.org/recipes#Category,28
3,http://example.org/recipes#Recipe,14
4,http://example.org/recipes#Allergen,14


Stardog agrees with the local check. Now use the language tags: the French menu, with prep times sorted **numerically** because they're `xsd:integer`:

In [18]:
select_df(f'''
PREFIX rec: <http://example.org/recipes#>
SELECT ?nom ?cuisine ?minutes
FROM <{RECIPES_GRAPH}>
WHERE {{
  ?r a rec:Recipe ; rec:name ?nom ; rec:cuisine ?cuisine ; rec:prepMinutes ?minutes .
  FILTER(lang(?nom) = "fr")
}}
ORDER BY ?minutes ?nom''')

,nom,cuisine,minutes
0,Pâtes au pesto,Italian,15
1,Galette à la farine de lupin,Mediterranean,20
2,Pancakes,American,20
3,Saumon grillé,Mediterranean,20
4,Spaghetti à la carbonara,Italian,20
5,Moules marinières,French,25
6,Pad thaï,Thai,25
7,Poulet sauté au wok,Chinese,25
8,Aiguillettes de poulet croustillantes,American,30
9,Pizza Margherita,Italian,30


And the blank-node ingredient lines, as a shopping list for Pad Thai:

In [19]:
select_df(f'''
PREFIX rec: <http://example.org/recipes#>
PREFIX recipe: <http://example.org/recipes/recipe/>
SELECT ?ingredient ?quantity ?unit
FROM <{RECIPES_GRAPH}>
WHERE {{
  recipe:pad-thai rec:hasIngredientLine ?line .
  ?line rec:ingredient/rec:name ?ingredient ; rec:quantity ?quantity ; rec:unit ?unit .
}}
ORDER BY ?ingredient''')

,ingredient,quantity,unit
0,Egg,2,piece
1,Fish Sauce,30,ml
2,Garlic,2,clove
3,Lime,1,piece
4,Peanuts,30,g
5,Rice Noodles,200,g
6,Shrimp,150,g
7,Tofu,100,g


### Step 7: export as JSON-LD

`conn.export()` writes a graph in any format, which is useful for handing data to a web app. Stardog builds the JSON-LD `@context` from the database's namespaces (Chapter 2):

In [20]:
jsonld = conn.export(content_type='application/ld+json', graph_uri=RECIPES_GRAPH).decode()
print(f'{len(jsonld):,} characters. The Pad Thai entry:\n')
start = jsonld.find('"@id" : "http://example.org/recipes/recipe/pad-thai"')
print(jsonld[jsonld.rfind('{', 0, start):start + 700], '\n…')

108,903 characters. The Pad Thai entry:

{
      "@id" : "http://example.org/recipes/recipe/pad-thai",
      "@type" : "http://example.org/recipes#Recipe",
      "http://example.org/recipes#cuisine" : {
        "@value" : "Thai"
      },
      "http://example.org/recipes#name" : [ {
        "@value" : "Pad Thai",
        "@language" : "en"
      }, {
        "@value" : "Pad thaï",
        "@language" : "fr"
      } ],
      "http://example.org/recipes#hasIngredient" : [ {
        "@id" : "http://example.org/recipes/ingredient/egg"
      }, {
        "@id" : "http://example.org/recipes/ingredient/tofu"
      }, {
        "@id" : "http://example.org/recipes/ingredient/peanuts"
      }, {
        "@id" : "http://example.org/recipes/ingredient 
…


## 3.8 Exercises

**Exercise 1.** Add a new recipe, *Masala Omelette* (Indian, 15 minutes, 1 serving, French name *Omelette masala*, ingredients egg and onion), by loading Turtle with `Raw` into `RECIPES_GRAPH`. Then query it back in French.

<details><summary>Solution</summary>

```python
new = '''
@prefix rec: <http://example.org/recipes#> .
@prefix recipe: <http://example.org/recipes/recipe/> .
@prefix ingredient: <http://example.org/recipes/ingredient/> .
recipe:masala-omelette a rec:Recipe ;
    rec:name "Masala Omelette"@en , "Omelette masala"@fr ;
    rec:cuisine "Indian" ; rec:prepMinutes 15 ; rec:servings 1 ;
    rec:hasIngredient ingredient:egg , ingredient:onion .
'''
with client.transaction(conn):
    conn.add(stardog.content.Raw(new.encode(), 'text/turtle'), graph_uri=RECIPES_GRAPH)

select_df(f'''PREFIX rec: <http://example.org/recipes#>
PREFIX recipe: <http://example.org/recipes/recipe/>
SELECT ?nom FROM <{RECIPES_GRAPH}> {{ recipe:masala-omelette rec:name ?nom FILTER(lang(?nom) = "fr") }}''')
```
Note `prepMinutes 15` without quotes, so it's an `xsd:integer`.
</details>

**Exercise 2.** In `TYPES_GRAPH`, sort by `ex:minutesText` **correctly** without changing the data.

<details><summary>Solution</summary>

Cast in the query:
```sparql
PREFIX ex: <http://example.org/demo#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?dish ?value FROM <urn:tutorial:ch03:types>
WHERE { ?dish ex:minutesText ?value } ORDER BY xsd:integer(?value)
```
It works, but casting on every query is slow and easy to forget. Fixing the data is better.
</details>

**Exercise 3.** Load `recipes.ttl` into `RECIPES_GRAPH` a second time **without** clearing, and count `rec:IngredientLine` nodes. What happened, and how do you fix it?

<details><summary>Solution</summary>

```python
with client.transaction(conn):
    conn.add(stardog.content.File(str(TTL_FILE)), graph_uri=RECIPES_GRAPH)
select_df(f'PREFIX rec: <http://example.org/recipes#> SELECT (COUNT(?l) AS ?lines) FROM <{RECIPES_GRAPH}> {{ ?l a rec:IngredientLine }}')
```
168 instead of 84: the blank-node lines were duplicated, while the recipes, ingredients and names were not. Fix: clear the graph before loading (Step 6), or give lines IRIs.
</details>

**Exercise 4.** Write TriG that puts the allergens in their own graph `urn:tutorial:ch03:allergens`, load it, and query allergen names with `FROM NAMED` + `GRAPH ?g` so the graph name shows up.

<details><summary>Solution</summary>

```python
trig = '''
@prefix rec: <http://example.org/recipes#> .
@prefix allergen: <http://example.org/recipes/allergen/> .
<urn:tutorial:ch03:allergens> {
  allergen:gluten a rec:Allergen ; rec:name "Gluten" .
  allergen:milk   a rec:Allergen ; rec:name "Milk" .
}'''
with client.transaction(conn):
    conn.add(stardog.content.Raw(trig.encode(), 'application/trig'))
select_df('''PREFIX rec: <http://example.org/recipes#>
SELECT ?g ?name FROM NAMED <urn:tutorial:ch03:allergens>
WHERE { GRAPH ?g { ?a rec:name ?name } }''')
```
</details>

In [21]:
# your answers here

## 3.9 Clean up

Remove every named graph this chapter created (they all start with `urn:tutorial:ch03`). The `recipes.ttl` file stays for the next chapters.

In [22]:
CLEAN_UP = True

graphs = select_df(f'''SELECT DISTINCT ?g WHERE {{ GRAPH ?g {{ ?s ?p ?o }}
                       FILTER(STRSTARTS(STR(?g), "{PREFIX}")) }}''')['g'].tolist()
if CLEAN_UP:
    with client.transaction(conn):
        for g in graphs:
            conn.clear(graph_uri=g)        # one named graph at a time, never clear() without graph_uri
    print('Removed:', *graphs, sep='\n  ')
conn.close()

Removed:
  urn:tutorial:ch03:types
  urn:tutorial:ch03:lang
  urn:tutorial:ch03:bnodes
  urn:tutorial:ch03:source:kitchen
  urn:tutorial:ch03:source:nutrition
  urn:tutorial:ch03:recipes


## Summary

- RDF has three kinds of terms: **IRIs** (global names), **literals** (values), and **blank nodes** (local, anonymous nodes).
- Give literals the right **datatype**. Numbers stored as text sort and compare wrongly, and comparing them silently drops rows. **Language tags** hold translations of the same text.
- Blank nodes suit small value-like structures, but **reloading duplicates them**. Load idempotently (clear, then load) or use IRIs.
- **Formats:** Turtle for people, N-Triples for bulk, JSON-LD for web apps, RDF/XML for legacy, TriG and N-Quads for named graphs.
- A database is a **dataset**: a default graph plus named graphs. `FROM` picks and merges graphs, and `FROM NAMED` + `GRAPH ?g` shows where data came from. Without `FROM`, Stardog reads the default graph only.
- Load with `conn.add(File | Raw | URL, graph_uri=…)` inside a transaction. **`client.transaction()`** keeps the connection usable after a failed load.
- The **Recipes dataset** (1,000+ triples) is now in `data/recipes/recipes.ttl`.

### Cheat sheet

```python
with client.transaction(conn):
    conn.clear(graph_uri=G)                                         # idempotent load
    conn.add(stardog.content.File('data.ttl'), graph_uri=G)         # or .nt, .jsonld, .ttl.gz, …
    conn.add(stardog.content.Raw(text, 'application/trig'))         # TriG: graphs from the file

conn.export(content_type='application/ld+json', graph_uri=G)        # dump a graph
```

```sparql
FILTER(lang(?name) = "fr")                       # pick a language
FROM <g1> FROM <g2>                              # merge graphs
FROM NAMED <g1> … GRAPH ?g { … }                 # keep track of the source graph
ORDER BY xsd:integer(?text)                      # cast when the data is badly typed
```

**Next: Chapter 4, SPARQL essentials.** SELECT, FILTER, OPTIONAL, UNION, ORDER BY and LIMIT on the Recipes graph, and a reusable `run_query()` helper backed by `.rq` files.